[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EddiazOS/LC_data_review/blob/main/05_Analisis_Reproducible/notebooks/02_nivel_2_variabilidad_espacial.ipynb)

# Notebook 02: Nivel 2 — Variabilidad Espacial y Microambiental
### Subsección 3.2 del Manuscrito: Pruebas de Hipótesis, Geografía, Perfiles y Figuras 2 y 3
**Proyecto LC-GC / Indoor Dust — Universidad de Cartagena & Universidad de Amberes**


In [ ]:
# Celda 1: Configuración de Entorno (Google Colab / Local) y Pre-requisitos
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("🚀 Entorno Google Colab detectado.")
    if not os.path.exists('LC_data_review'):
        !git clone https://github.com/EddiazOS/LC_data_review.git
    if os.path.exists('LC_data_review/05_Analisis_Reproducible/datos_curados'):
        DATA_DIR = 'LC_data_review/05_Analisis_Reproducible/datos_curados'
        OUT_TABLES = 'LC_data_review/05_Analisis_Reproducible/outputs/tablas'
        OUT_FIGS = 'LC_data_review/05_Analisis_Reproducible/outputs/figuras'
    else:
        DATA_DIR = '../datos_curados'
        OUT_TABLES = '../outputs/tablas'
        OUT_FIGS = '../outputs/figuras'
else:
    print("💻 Entorno Local detectado.")
    DATA_DIR = '../datos_curados' if os.path.exists('../datos_curados') else '05_Analisis_Reproducible/datos_curados'
    OUT_TABLES = '../outputs/tablas' if os.path.exists('../outputs/tablas') else '05_Analisis_Reproducible/outputs/tablas'
    OUT_FIGS = '../outputs/figuras' if os.path.exists('../outputs/figuras') else '05_Analisis_Reproducible/outputs/figuras'

os.makedirs(OUT_TABLES, exist_ok=True)
os.makedirs(OUT_FIGS, exist_ok=True)
print(f"Directorio de datos activo: {DATA_DIR}")

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'
print("Librerías cargadas correctamente.")


In [ ]:
# Celda 2: Auto-Carga de Datos Curados
df_muestras = pd.read_csv(os.path.join(DATA_DIR, 'muestras_metadata.csv'))
df_compuestos = pd.read_csv(os.path.join(DATA_DIR, 'compuestos_metadata.csv'))
df_conc = pd.read_csv(os.path.join(DATA_DIR, 'concentraciones_ng_g.csv'))
df_det = pd.read_csv(os.path.join(DATA_DIR, 'estado_deteccion.csv'))

acronyms = list(df_compuestos['acronym'])
fam_map = dict(zip(df_compuestos['acronym'], df_compuestos['family']))

aps_cols = [c for c in acronyms if fam_map[c] == 'APs']
pfrs_cols = [c for c in acronyms if fam_map[c] == 'PFRs']
phs_cols = [c for c in acronyms if fam_map[c] == 'PHs']

df_muestras['Sum_APs'] = df_conc[aps_cols].sum(axis=1)
df_muestras['Sum_PFRs'] = df_conc[pfrs_cols].sum(axis=1)
df_muestras['Sum_PHs'] = df_conc[phs_cols].sum(axis=1)
df_muestras['Sum_Total'] = df_muestras['Sum_APs'] + df_muestras['Sum_PFRs'] + df_muestras['Sum_PHs']

print("Datos y sumatorias preparadas.")


In [ ]:
# Celda 3: Contraste Geográfico entre Centros Urbanos (Bogotá vs Medellín vs Cartagena)
print("=== CONTRASTE GEOGRÁFICO DE SUMATORIAS ===")
for metric in ['Sum_Total', 'Sum_PHs', 'Sum_APs', 'Sum_PFRs']:
    groups = [df_muestras[df_muestras['city'] == c][metric].values for c in ['Bogota', 'Medellin', 'Cartagena']]
    
    f_stat, f_p = stats.f_oneway(*groups)
    kw_stat, kw_p = stats.kruskal(*groups)
    
    print(f"{metric:<10}: ANOVA F = {f_stat:.4f} (p = {f_p:.4f}) | Kruskal-Wallis H = {kw_stat:.4f} (p = {kw_p:.4f})")


In [ ]:
# Celda 4: Contraste Residencial Directo (Hogares Bogotá vs Medellín)
df_homes = df_muestras[df_muestras['microenvironment'] == 'House'].copy()
df_homes_conc = df_conc.loc[df_homes.index].copy()

bog_idxs = df_homes[df_homes['city'] == 'Bogota'].index
med_idxs = df_homes[df_homes['city'] == 'Medellin'].index

print(f"Hogares evaluados: Bogotá (n = {len(bog_idxs)}), Medellín (n = {len(med_idxs)})\n")

for target in ['Sum_PFRs', 'TBOEP', 'TCIPP', 'Sum_Total', 'Sum_APs', 'Sum_PHs']:
    if target.startswith('Sum_'):
        vals_bog = df_homes.loc[bog_idxs, target].values
        vals_med = df_homes.loc[med_idxs, target].values
    else:
        vals_bog = df_homes_conc.loc[bog_idxs, target].values
        vals_med = df_homes_conc.loc[med_idxs, target].values
        
    mwu_stat, mwu_p = stats.mannwhitneyu(vals_bog, vals_med, alternative='two-sided')
    print(f"{target:<10}: Bogota Med = {np.median(vals_bog):,.1f} vs Medellin Med = {np.median(vals_med):,.1f} | MWU p = {mwu_p:.4f}")


In [ ]:
# Celda 5: Generación de la Figura 2 (Geográfica: Panel A + Panel B)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

order_city = ['Bogota', 'Medellin', 'Cartagena']
palette_city = {'Bogota': '#34495E', 'Medellin': '#16A085', 'Cartagena': '#D35400'}

df_muestras['log10_Total'] = np.log10(df_muestras['Sum_Total'])
sns.boxplot(data=df_muestras, x='city', y='log10_Total', order=order_city, palette=palette_city, width=0.45, ax=ax1)
sns.stripplot(data=df_muestras, x='city', y='log10_Total', order=order_city, color='black', alpha=0.6, jitter=0.2, ax=ax1)
ax1.set_title('(a) Total Additives by Urban Center', fontweight='bold')
ax1.set_xlabel('City', fontweight='bold')
ax1.set_ylabel('Total Additives (log10 ng/g)', fontweight='bold')
ax1.grid(axis='y', linestyle='--', alpha=0.4)

df_homes_plot = df_homes.copy()
df_homes_plot['TBOEP'] = df_homes_conc['TBOEP']
df_homes_plot['TCIPP'] = df_homes_conc['TCIPP']

df_panel_b = []
for idx, row in df_homes_plot.iterrows():
    c = row['city']
    df_panel_b.append({'City': c, 'Metric': 'Sum_PFRs', 'Conc': row['Sum_PFRs']})
    df_panel_b.append({'City': c, 'Metric': 'TBOEP', 'Conc': row['TBOEP']})
    df_panel_b.append({'City': c, 'Metric': 'TCIPP', 'Conc': row['TCIPP']})
df_panel_b = pd.DataFrame(df_panel_b)
df_panel_b['log10_Conc'] = np.log10(df_panel_b['Conc'])

sns.boxplot(data=df_panel_b, x='Metric', y='log10_Conc', hue='City', palette={'Bogota': '#34495E', 'Medellin': '#16A085'}, ax=ax2, width=0.55)
ax2.set_title('(b) Intra-residential Flame Retardants (Homes)', fontweight='bold')
ax2.set_xlabel('Metric', fontweight='bold')
ax2.set_ylabel('Concentration (log10 ng/g)', fontweight='bold')
ax2.grid(axis='y', linestyle='--', alpha=0.4)

fig2_path = os.path.join(OUT_FIGS, 'figura_2_geografica_completa.png')
plt.savefig(fig2_path)
plt.show()
print(f"Figura 2 guardada en: {fig2_path}")
